# Volatility-Quantile Regime Detection: Teaching and Research Notebook

**Thesis:** *Regime-Conditioned Multivariate Motif Discovery In Financial Time Series: A Reproducible Empirical Benchmark Under Nonstationarity*

Financial time series are nonstationary: their distributional properties change across time as liquidity, leverage, macro conditions, news intensity, and market participation evolve. One of the most visible forms of this nonstationarity is volatility clustering, where large price changes tend to be followed by large price changes and calm periods tend to persist.

This matters for motif discovery. A shape pattern observed during a calm market can have a very different meaning from a visually similar pattern observed during a high-volatility market. Blindly comparing motifs across different volatility states can mix structurally different market conditions and make empirical conclusions less interpretable.

This notebook implements the first regime detection method only: **Volatility-Quantile Regime Detection**. It is a transparent baseline that segments BTCUSDT and ETHUSDT time series into low, medium, and high volatility regimes using rolling realized volatility and empirical quantile thresholds. The resulting labels are saved for later Matrix Profile and LoCoMotif experiments.

This notebook does **not** implement HMMs and does **not** run motif discovery.

## 1. Imports and Configuration

The quantile method is deterministic and model-free, so no machine learning library is required. All discovery, cleaning, feature engineering, plotting, and export steps are handled with standard scientific Python libraries.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"C:\Users\learn\OneDrive\Desktop\Final Masters Thesis")
DATA_DIR = PROJECT_ROOT / "final_dataset"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "Regime Studies"
RESULT_DIR = PROJECT_ROOT / "reports" / "results" / "regime_studies" / "01_volatility_quantile_regimes"

FIGURE_DIR = RESULT_DIR / "figures"
TABLE_DIR = RESULT_DIR / "tables"
REGIME_LABEL_DIR = RESULT_DIR / "regime_labels"
CONFIG_DIR = RESULT_DIR / "config"
LOG_DIR = RESULT_DIR / "logs"

for directory in [FIGURE_DIR, TABLE_DIR, REGIME_LABEL_DIR, CONFIG_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TARGET_SYMBOLS = ["BTCUSDT", "ETHUSDT"]
USE_YEAR_FILTER = True
YEAR_TO_ANALYZE = 2025
ROLLING_WINDOW_OVERRIDE = None
LOWER_QUANTILE = 0.33
UPPER_QUANTILE = 0.66

SUPPORTED_EXTENSIONS = {".parquet", ".csv", ".feather", ".pkl", ".pickle"}

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print(f"Data directory: {DATA_DIR}")
print(f"Result directory: {RESULT_DIR}")

Data directory: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\final_dataset
Result directory: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes


## 2. Data Discovery

The project data may be organized in subfolders and may use different timestamp, price, volume, and symbol column names. The helper functions below search recursively, inspect candidate files, detect required columns, and load BTCUSDT and ETHUSDT separately.

The loader prefers clean crypto files for the requested assets, while remaining robust to multi-symbol files and alternative formats.

In [2]:
def discover_candidate_files(data_dir):
    """Find supported data files recursively and rank likely crypto OHLCV sources first."""
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory does not exist: {data_dir}")

    records = []
    preferred_keywords = ["btcusdt", "ethusdt", "btc", "eth", "crypto", "final", "clean", "processed"]
    avoid_keywords = ["metadata", "readme", "requirements", "missing_data", "download_log"]

    for path in data_dir.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        name_lower = path.name.lower()
        full_lower = str(path).lower()
        score = 0
        reasons = []

        if path.suffix.lower() == ".parquet":
            score += 10
            reasons.append("parquet")
        if "processed" in full_lower:
            score += 20
            reasons.append("processed")
        if "crypto" in full_lower:
            score += 15
            reasons.append("crypto")
        if "features" in full_lower:
            score += 5
            reasons.append("features_available")
        if "raw" in full_lower:
            score -= 15
            reasons.append("raw_lower_priority")
        if "1h" in name_lower or "\\1h\\" in full_lower or "/1h/" in full_lower:
            score += 18
            reasons.append("1h_preferred")
        elif "15m" in name_lower or "\\15m\\" in full_lower or "/15m/" in full_lower:
            score += 12
            reasons.append("15m")
        elif "5m" in name_lower or "\\5m\\" in full_lower or "/5m/" in full_lower:
            score += 8
            reasons.append("5m")
        elif "1d" in name_lower or "\\1d\\" in full_lower or "/1d/" in full_lower:
            score += 6
            reasons.append("1d")
        elif "1m" in name_lower or "\\1m\\" in full_lower or "/1m/" in full_lower:
            score += 2
            reasons.append("1m_large_but_valid")

        for keyword in preferred_keywords:
            if keyword in full_lower:
                score += 4
                reasons.append(f"keyword:{keyword}")
        for keyword in avoid_keywords:
            if keyword in full_lower:
                score -= 100
                reasons.append(f"avoid:{keyword}")

        records.append({
            "path": path,
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "discovery_score": score,
            "discovery_reasons": ";".join(reasons),
        })

    candidates = pd.DataFrame(records)
    if candidates.empty:
        expected = ", ".join(sorted(SUPPORTED_EXTENSIONS))
        raise FileNotFoundError(f"No candidate files found under {data_dir}. Expected extensions: {expected}")

    candidates = candidates.sort_values(["discovery_score", "size_bytes"], ascending=[False, True]).reset_index(drop=True)
    return candidates


def preview_file(path, max_rows=5000):
    """Read a small preview from a supported file. Returns an empty DataFrame if preview fails."""
    path = Path(path)
    ext = path.suffix.lower()
    try:
        if ext == ".csv":
            return pd.read_csv(path, nrows=max_rows)
        if ext == ".feather":
            return pd.read_feather(path).head(max_rows)
        if ext in {".pkl", ".pickle"}:
            obj = pd.read_pickle(path)
            return obj.head(max_rows) if isinstance(obj, pd.DataFrame) else pd.DataFrame()
        if ext == ".parquet":
            try:
                import pyarrow.parquet as pq

                parquet_file = pq.ParquetFile(path)
                if parquet_file.num_row_groups > 0:
                    table = parquet_file.read_row_group(0)
                    return table.to_pandas().head(max_rows)
            except Exception:
                pass
            return pd.read_parquet(path).head(max_rows)
    except Exception as exc:
        warnings.warn(f"Could not preview {path}: {exc}")
        return pd.DataFrame()

    return pd.DataFrame()


def _normalise_column_name(column):
    return str(column).strip().lower().replace(" ", "_")


def detect_timestamp_column(df):
    candidates = ["timestamp", "datetime", "date", "open_time", "close_time", "time", "timestamp_utc"]
    normalised = {_normalise_column_name(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalised:
            return normalised[candidate]
    for col in df.columns:
        name = _normalise_column_name(col)
        if "time" in name or "date" in name:
            return col
    return None


def detect_close_column(df):
    candidates = ["close", "close_price", "adj_close", "last", "price"]
    normalised = {_normalise_column_name(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalised:
            return normalised[candidate]
    for col in df.columns:
        name = _normalise_column_name(col)
        if name.endswith("close") or "close" in name:
            return col
    return None


def detect_volume_column(df):
    candidates = ["volume", "base_volume", "quote_volume", "vol"]
    normalised = {_normalise_column_name(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalised:
            return normalised[candidate]
    for col in df.columns:
        if "volume" in _normalise_column_name(col):
            return col
    return None


def detect_symbol_column(df):
    candidates = ["symbol", "asset", "ticker", "instrument", "pair"]
    normalised = {_normalise_column_name(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalised:
            return normalised[candidate]
    return None


def read_full_file(path, columns=None):
    path = Path(path)
    ext = path.suffix.lower()
    if ext == ".csv":
        return pd.read_csv(path, usecols=columns)
    if ext == ".feather":
        df = pd.read_feather(path)
        return df[columns] if columns else df
    if ext in {".pkl", ".pickle"}:
        df = pd.read_pickle(path)
        return df[columns] if columns else df
    if ext == ".parquet":
        return pd.read_parquet(path, columns=columns)
    raise ValueError(f"Unsupported file extension: {path.suffix}")


def _target_symbol_from_filename(path, target_symbols):
    name = Path(path).name.upper()
    for symbol in target_symbols:
        if symbol.upper() in name:
            return symbol
    for symbol in target_symbols:
        base = symbol.replace("USDT", "")
        if base in name:
            return symbol
    return None


def _score_file_for_symbol(row, preview, target_symbol, timestamp_col, close_col, symbol_col):
    score = float(row["discovery_score"])
    path_upper = str(row["path"]).upper()
    if target_symbol in path_upper:
        score += 120
    elif target_symbol.replace("USDT", "") in path_upper:
        score += 45
    if timestamp_col is not None and close_col is not None:
        score += 100
    if symbol_col is not None and not preview.empty:
        values = preview[symbol_col].astype(str).str.upper().unique()[:100]
        if target_symbol in set(values):
            score += 140
    elif _target_symbol_from_filename(row["path"], [target_symbol]) == target_symbol:
        score += 80
    if preview.shape[0] > 0:
        score += min(preview.shape[0] / 1000, 5)
    return score


def load_asset_data(data_dir, target_symbols):
    """Load one clean DataFrame per target symbol and return discovery logs."""
    candidate_files = discover_candidate_files(data_dir)
    inspection_records = []
    valid_records = []

    for _, row in candidate_files.iterrows():
        path = row["path"]
        preview = preview_file(path)
        timestamp_col = detect_timestamp_column(preview) if not preview.empty else None
        close_col = detect_close_column(preview) if not preview.empty else None
        volume_col = detect_volume_column(preview) if not preview.empty else None
        symbol_col = detect_symbol_column(preview) if not preview.empty else None

        inspected = {
            "path": str(path),
            "file_name": row["file_name"],
            "extension": row["extension"],
            "size_bytes": int(row["size_bytes"]),
            "discovery_score": float(row["discovery_score"]),
            "preview_rows": int(len(preview)),
            "preview_columns": "|".join(map(str, preview.columns)) if not preview.empty else "",
            "timestamp_col": timestamp_col,
            "close_col": close_col,
            "volume_col": volume_col,
            "symbol_col": symbol_col,
            "usable": bool(timestamp_col is not None and close_col is not None),
            "notes": row["discovery_reasons"],
        }
        inspection_records.append(inspected)

        if timestamp_col is None or close_col is None:
            continue

        for symbol in target_symbols:
            score = _score_file_for_symbol(row, preview, symbol, timestamp_col, close_col, symbol_col)
            possible = False
            if symbol_col is not None and not preview.empty:
                possible = symbol in set(preview[symbol_col].astype(str).str.upper().unique())
            if _target_symbol_from_filename(path, [symbol]) == symbol:
                possible = True
            if symbol in str(path).upper():
                possible = True

            if possible:
                valid_records.append({
                    "target_symbol": symbol,
                    "path": path,
                    "score": score,
                    "timestamp_col": timestamp_col,
                    "close_col": close_col,
                    "volume_col": volume_col,
                    "symbol_col": symbol_col,
                    "preview_columns": list(preview.columns),
                })

    discovery_log = pd.DataFrame(inspection_records)
    if not valid_records:
        expected = ", ".join(sorted(SUPPORTED_EXTENSIONS))
        raise RuntimeError(
            f"No usable BTCUSDT/ETHUSDT files found under {data_dir}. "
            f"Searched recursively for: {expected}. Check logs/data_discovery_log.csv after rerun."
        )

    selected = {}
    for symbol in target_symbols:
        symbol_records = [record for record in valid_records if record["target_symbol"] == symbol]
        if not symbol_records:
            warnings.warn(f"No usable data found for {symbol}; continuing with available assets.")
            continue

        best = sorted(symbol_records, key=lambda record: record["score"], reverse=True)[0]
        required_columns = [best["timestamp_col"], best["close_col"]]
        optional_columns = [best["volume_col"], best["symbol_col"]]
        columns = []
        for col in required_columns + optional_columns:
            if col is not None and col not in columns:
                columns.append(col)

        raw = read_full_file(best["path"], columns=columns)

        if best["symbol_col"] is not None:
            raw = raw[raw[best["symbol_col"]].astype(str).str.upper() == symbol].copy()
        else:
            raw = raw.copy()
            raw["symbol"] = symbol
            best["symbol_col"] = "symbol"

        rename_map = {
            best["timestamp_col"]: "timestamp",
            best["close_col"]: "close",
            best["symbol_col"]: "symbol",
        }
        if best["volume_col"] is not None:
            rename_map[best["volume_col"]] = "volume"
        raw = raw.rename(columns=rename_map)

        keep_columns = ["timestamp", "close"]
        if "volume" in raw.columns:
            keep_columns.append("volume")
        if "symbol" in raw.columns:
            keep_columns.append("symbol")

        selected[symbol] = {
            "data": raw[keep_columns].copy(),
            "input_file": str(best["path"]),
            "score": best["score"],
        }

    asset_data = {symbol: record["data"] for symbol, record in selected.items()}
    input_files_used = {symbol: record["input_file"] for symbol, record in selected.items()}

    if len(asset_data) == 1:
        warnings.warn("Only one target asset was found. The notebook will continue with the available asset.")
    if not asset_data:
        raise RuntimeError("No target assets could be loaded after candidate inspection.")

    return asset_data, input_files_used, discovery_log


asset_data, input_files_used, discovery_log = load_asset_data(DATA_DIR, TARGET_SYMBOLS)
discovery_log.to_csv(LOG_DIR / "data_discovery_log.csv", index=False)

print("Loaded assets:")
for symbol, df in asset_data.items():
    print(f"- {symbol}: {len(df):,} rows from {input_files_used[symbol]}")

print(f"Discovery log saved to: {LOG_DIR / 'data_discovery_log.csv'}")

Loaded assets:
- BTCUSDT: 52,577 rows from C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\final_dataset\processed\crypto\1h\BTCUSDT_1h_2020_2025.parquet
- ETHUSDT: 52,577 rows from C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\final_dataset\processed\crypto\1h\ETHUSDT_1h_2020_2025.parquet
Discovery log saved to: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\logs\data_discovery_log.csv


## 3. Data Cleaning

Each asset is cleaned independently. The cleaning step standardizes timestamps, sorts observations, removes duplicate timestamps, keeps only positive close prices, reports coverage, and optionally filters to the thesis analysis year.

If 2025 exists in the data, this notebook uses 2025 by default to keep the regime study focused and computationally practical. If 2025 is unavailable, it falls back to the full available range and raises a warning.

In [3]:
def convert_timestamp(series):
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        median_value = numeric.dropna().median()
        if pd.isna(median_value):
            return pd.to_datetime(series, errors="coerce", utc=True)
        if median_value > 1e17:
            unit = "ns"
        elif median_value > 1e14:
            unit = "us"
        elif median_value > 1e11:
            unit = "ms"
        else:
            unit = "s"
        return pd.to_datetime(numeric, errors="coerce", unit=unit, utc=True)
    return pd.to_datetime(series, errors="coerce", utc=True)


def infer_sampling_frequency(timestamps):
    clean_times = pd.Series(timestamps).dropna().sort_values()
    diffs = clean_times.diff().dropna()
    if diffs.empty:
        return {"median_delta": pd.NaT, "frequency_label": "unknown", "periods_per_year": np.nan}

    median_delta = diffs.median()
    minutes = median_delta / pd.Timedelta(minutes=1)

    if minutes <= 2:
        label = "minute_level"
        periods_per_year = 365 * 24 * 60
    elif minutes <= 10:
        label = "five_minute_level"
        periods_per_year = 365 * 24 * 12
    elif minutes <= 20:
        label = "fifteen_minute_level"
        periods_per_year = 365 * 24 * 4
    elif minutes <= 90:
        label = "hourly"
        periods_per_year = 365 * 24
    elif minutes <= 60 * 30:
        label = "daily_or_intraday_gap"
        periods_per_year = 365
    else:
        label = "low_frequency_or_irregular"
        periods_per_year = np.nan

    return {"median_delta": median_delta, "frequency_label": label, "periods_per_year": periods_per_year}


def clean_asset_frame(df, asset, use_year_filter=True, year_to_analyze=2025):
    cleaned = df.copy()
    cleaned["timestamp"] = convert_timestamp(cleaned["timestamp"])
    cleaned["close"] = pd.to_numeric(cleaned["close"], errors="coerce")
    if "volume" in cleaned.columns:
        cleaned["volume"] = pd.to_numeric(cleaned["volume"], errors="coerce")
    if "symbol" not in cleaned.columns:
        cleaned["symbol"] = asset

    missing_close_before = int(cleaned["close"].isna().sum())
    cleaned = cleaned.dropna(subset=["timestamp", "close"])
    cleaned = cleaned[cleaned["close"] > 0].copy()
    cleaned = cleaned.sort_values("timestamp").drop_duplicates("timestamp", keep="last").reset_index(drop=True)

    full_start = cleaned["timestamp"].min()
    full_end = cleaned["timestamp"].max()
    used_year_filter = False

    if use_year_filter:
        year_mask = cleaned["timestamp"].dt.year == year_to_analyze
        if year_mask.any():
            cleaned = cleaned.loc[year_mask].reset_index(drop=True)
            used_year_filter = True
        else:
            warnings.warn(f"{asset}: year {year_to_analyze} was not found; using full range {full_start} to {full_end}.")

    frequency_info = infer_sampling_frequency(cleaned["timestamp"])

    report = {
        "asset": asset,
        "start_time": cleaned["timestamp"].min(),
        "end_time": cleaned["timestamp"].max(),
        "n_rows": len(cleaned),
        "missing_close_before_cleaning": missing_close_before,
        "missing_close_after_cleaning": int(cleaned["close"].isna().sum()),
        "median_delta": frequency_info["median_delta"],
        "frequency_label": frequency_info["frequency_label"],
        "periods_per_year": frequency_info["periods_per_year"],
        "used_year_filter": used_year_filter,
    }
    return cleaned, report


cleaned_asset_data = {}
cleaning_reports = []
frequency_info_by_asset = {}

for asset, df in asset_data.items():
    cleaned, report = clean_asset_frame(df, asset, USE_YEAR_FILTER, YEAR_TO_ANALYZE)
    cleaned_asset_data[asset] = cleaned
    cleaning_reports.append(report)
    frequency_info_by_asset[asset] = {
        "frequency_label": report["frequency_label"],
        "periods_per_year": report["periods_per_year"],
        "median_delta": report["median_delta"],
    }

cleaning_report_df = pd.DataFrame(cleaning_reports)
display(cleaning_report_df)
cleaning_report_df.to_csv(TABLE_DIR / "data_cleaning_report_by_asset.csv", index=False)

,asset,start_time,end_time,n_rows,missing_close_before_cleaning,missing_close_after_cleaning,median_delta,frequency_label,periods_per_year,used_year_filter
0,BTCUSDT,2025-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,8760,0,0,0 days 01:00:00,hourly,8760,True
1,ETHUSDT,2025-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,8760,0,0,0 days 01:00:00,hourly,8760,True


## 4. Feature Engineering

For each asset, the notebook computes log prices, log returns, absolute log returns, and rolling realized volatility. Regime assignment uses only contemporaneously available rolling history:

`sigma_t = std(r_{t-w+1}, ..., r_t)`

This avoids look-ahead leakage because no future returns are used to label the current observation.

In [4]:
def default_rolling_window(frequency_label):
    if frequency_label == "minute_level":
        return 60
    if frequency_label == "hourly":
        return 24
    if frequency_label == "daily_or_intraday_gap":
        return 20
    return 60


def add_return_and_volatility_features(df, frequency_info, rolling_window_override=None):
    featured = df.copy()
    featured["log_close"] = np.log(featured["close"])
    featured["log_return"] = featured["log_close"].diff()
    featured["abs_log_return"] = featured["log_return"].abs()

    window = rolling_window_override or default_rolling_window(frequency_info["frequency_label"])
    featured["rolling_realized_vol"] = featured["log_return"].rolling(window=window, min_periods=window).std()

    periods_per_year = frequency_info.get("periods_per_year", np.nan)
    if pd.notna(periods_per_year) and periods_per_year > 0:
        featured["rolling_realized_vol_annualized_like"] = featured["rolling_realized_vol"] * np.sqrt(periods_per_year)
    else:
        featured["rolling_realized_vol_annualized_like"] = np.nan

    return featured, int(window)


featured_asset_data = {}
rolling_windows_used = {}

for asset, df in cleaned_asset_data.items():
    featured, window = add_return_and_volatility_features(
        df,
        frequency_info_by_asset[asset],
        rolling_window_override=ROLLING_WINDOW_OVERRIDE,
    )
    featured_asset_data[asset] = featured
    rolling_windows_used[asset] = window
    print(f"{asset}: rolling window = {window} observations; non-null rolling vol = {featured['rolling_realized_vol'].notna().sum():,}")

BTCUSDT: rolling window = 24 observations; non-null rolling vol = 8,736
ETHUSDT: rolling window = 24 observations; non-null rolling vol = 8,736


## 5. Quantile Regime Definition

Volatility-quantile regimes split the empirical distribution of rolling realized volatility into three states:

- **Low-volatility regime:** `rolling_vol <= 33rd percentile`
- **Medium-volatility regime:** `33rd percentile < rolling_vol <= 66th percentile`
- **High-volatility regime:** `rolling_vol > 66th percentile`

This is a transparent baseline because it is deterministic, easy to interpret, model-free, directly connected to volatility clustering, and useful before motif discovery. Motifs should not be blindly compared across very different volatility states, so these labels provide a simple first way to condition later Matrix Profile and LoCoMotif experiments on comparable market states.

In [5]:
def assign_volatility_quantile_regimes(df, vol_col="rolling_realized_vol", lower_q=0.33, upper_q=0.66):
    """Assign low, medium, and high volatility regimes using empirical quantiles."""
    if vol_col not in df.columns:
        raise KeyError(f"Missing volatility column: {vol_col}")

    assigned = df.copy()
    valid_vol = assigned[vol_col].dropna()
    if valid_vol.empty:
        raise ValueError("Cannot assign quantile regimes because rolling volatility is entirely missing.")

    q33 = float(valid_vol.quantile(lower_q))
    q66 = float(valid_vol.quantile(upper_q))

    assigned["vol_q33"] = q33
    assigned["vol_q66"] = q66
    assigned["regime_quantile_id"] = pd.Series(pd.NA, index=assigned.index, dtype="Int64")
    assigned["regime_quantile_label"] = "insufficient_history"

    valid_mask = assigned[vol_col].notna()
    low_mask = valid_mask & (assigned[vol_col] <= q33)
    medium_mask = valid_mask & (assigned[vol_col] > q33) & (assigned[vol_col] <= q66)
    high_mask = valid_mask & (assigned[vol_col] > q66)

    assigned.loc[low_mask, "regime_quantile_id"] = 0
    assigned.loc[medium_mask, "regime_quantile_id"] = 1
    assigned.loc[high_mask, "regime_quantile_id"] = 2

    assigned.loc[low_mask, "regime_quantile_label"] = "low_vol"
    assigned.loc[medium_mask, "regime_quantile_label"] = "medium_vol"
    assigned.loc[high_mask, "regime_quantile_label"] = "high_vol"

    return assigned, {"vol_q33": q33, "vol_q66": q66}


regime_asset_data = {}
quantile_thresholds = {}

for asset, df in featured_asset_data.items():
    assigned, thresholds = assign_volatility_quantile_regimes(
        df,
        vol_col="rolling_realized_vol",
        lower_q=LOWER_QUANTILE,
        upper_q=UPPER_QUANTILE,
    )
    regime_asset_data[asset] = assigned
    quantile_thresholds[asset] = thresholds
    print(f"{asset}: q33={thresholds['vol_q33']:.8f}, q66={thresholds['vol_q66']:.8f}")

BTCUSDT: q33=0.00300329, q66=0.00458023
ETHUSDT: q33=0.00519710, q66=0.00761209


## 6. Regime Summary Tables

The summary tables describe the distributional properties of each regime and whether the labels form persistent blocks. A regime run is a consecutive block of the same quantile regime.

In [6]:
def summarize_regimes(df, asset):
    valid = df[df["regime_quantile_label"] != "insufficient_history"].copy()
    total = len(valid)
    rows = []
    for label, group in valid.groupby("regime_quantile_label", sort=False):
        rows.append({
            "asset": asset,
            "regime_quantile_label": label,
            "n_observations": int(len(group)),
            "percentage_observations": float(len(group) / total * 100) if total else np.nan,
            "start_time": group["timestamp"].min(),
            "end_time": group["timestamp"].max(),
            "mean_close": float(group["close"].mean()),
            "mean_log_return": float(group["log_return"].mean()),
            "std_log_return": float(group["log_return"].std()),
            "mean_abs_log_return": float(group["abs_log_return"].mean()),
            "mean_rolling_vol": float(group["rolling_realized_vol"].mean()),
            "median_rolling_vol": float(group["rolling_realized_vol"].median()),
            "min_rolling_vol": float(group["rolling_realized_vol"].min()),
            "max_rolling_vol": float(group["rolling_realized_vol"].max()),
        })
    return pd.DataFrame(rows).sort_values(["asset", "regime_quantile_label"]).reset_index(drop=True)


def summarize_regime_runs(df, asset):
    valid = df[df["regime_quantile_label"] != "insufficient_history"].copy()
    if valid.empty:
        return pd.DataFrame()

    valid["run_id"] = (valid["regime_quantile_label"] != valid["regime_quantile_label"].shift()).cumsum()
    runs = (
        valid.groupby(["run_id", "regime_quantile_label"], as_index=False)
        .agg(
            run_length=("regime_quantile_label", "size"),
            start_time=("timestamp", "min"),
            end_time=("timestamp", "max"),
        )
    )

    summary = (
        runs.groupby("regime_quantile_label")
        .agg(
            number_of_runs=("run_length", "count"),
            mean_run_length=("run_length", "mean"),
            median_run_length=("run_length", "median"),
            max_run_length=("run_length", "max"),
        )
        .reset_index()
    )
    summary.insert(0, "asset", asset)
    return summary.sort_values(["asset", "regime_quantile_label"]).reset_index(drop=True)


regime_summary_tables = []
run_summary_tables = []
for asset, df in regime_asset_data.items():
    regime_summary_tables.append(summarize_regimes(df, asset))
    run_summary_tables.append(summarize_regime_runs(df, asset))

regime_summary_by_asset = pd.concat(regime_summary_tables, ignore_index=True)
regime_run_summary_by_asset = pd.concat(run_summary_tables, ignore_index=True)

regime_summary_path = TABLE_DIR / "regime_summary_by_asset.csv"
run_summary_path = TABLE_DIR / "regime_run_summary_by_asset.csv"
regime_summary_by_asset.to_csv(regime_summary_path, index=False)
regime_run_summary_by_asset.to_csv(run_summary_path, index=False)

display(regime_summary_by_asset)
display(regime_run_summary_by_asset)
print(f"Saved: {regime_summary_path}")
print(f"Saved: {run_summary_path}")

,asset,regime_quantile_label,n_observations,percentage_observations,start_time,end_time,mean_close,mean_log_return,std_log_return,mean_abs_log_return,mean_rolling_vol,median_rolling_vol,min_rolling_vol,max_rolling_vol
0,BTCUSDT,high_vol,2970,33.997253,2025-01-06 21:00:00+00:00,2025-12-30 01:00:00+00:00,96323.229781,-9.350976e-07,0.006774,0.004496,0.006573,0.005887,0.004581,0.017149
1,BTCUSDT,low_vol,2883,33.001374,2025-01-02 01:00:00+00:00,2025-12-31 23:00:00+00:00,105377.785040,6.397506e-05,0.002380,0.001764,0.002213,0.002306,0.000657,0.003003
2,BTCUSDT,medium_vol,2883,33.001374,2025-01-02 00:00:00+00:00,2025-12-30 08:00:00+00:00,103416.179036,-8.945620e-05,0.003941,0.002899,0.003734,0.003700,0.003003,0.004580
3,ETHUSDT,high_vol,2970,33.997253,2025-01-07 21:00:00+00:00,2025-12-19 18:00:00+00:00,2959.315074,-8.547048e-05,0.010929,0.007241,0.010510,0.009468,0.007613,0.032130
4,ETHUSDT,low_vol,2883,33.001374,2025-01-02 00:00:00+00:00,2025-12-31 23:00:00+00:00,3147.549365,1.908649e-04,0.004178,0.003109,0.003962,0.004146,0.001273,0.005197
5,ETHUSDT,medium_vol,2883,33.001374,2025-01-07 14:00:00+00:00,2025-12-30 01:00:00+00:00,3094.481377,-1.454583e-04,0.006460,0.004698,0.006323,0.006266,0.005197,0.007612


,asset,regime_quantile_label,number_of_runs,mean_run_length,median_run_length,max_run_length
0,BTCUSDT,high_vol,121,24.545455,17.0,147
1,BTCUSDT,low_vol,134,21.514925,11.0,150
2,BTCUSDT,medium_vol,244,11.815574,7.0,98
3,ETHUSDT,high_vol,135,22.000000,20.0,118
4,ETHUSDT,low_vol,161,17.906832,7.0,210
5,ETHUSDT,medium_vol,280,10.296429,6.0,69


Saved: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\tables\regime_summary_by_asset.csv
Saved: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\tables\regime_run_summary_by_asset.csv


## 7. Visualizations

Each asset receives seven thesis-ready matplotlib figures:

A. Close price over time  
B. Log returns over time  
C. Rolling realized volatility with q33 and q66 thresholds  
D. Close price colored by quantile regime  
E. Rolling volatility colored by quantile regime  
F. Regime distribution bar chart  
G. Regime timeline strip

In [7]:
REGIME_COLORS = {
    "low_vol": "tab:green",
    "medium_vol": "tab:orange",
    "high_vol": "tab:red",
    "insufficient_history": "lightgray",
}
REGIME_ORDER = ["low_vol", "medium_vol", "high_vol"]


def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_close_price(df, asset):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["timestamp"], df["close"], linewidth=1.0, color="black")
    ax.set_title(f"{asset}: Close Price Over Time")
    ax.set_xlabel("Time")
    ax.set_ylabel("Close price")
    save_figure(fig, FIGURE_DIR / f"{asset}_01_close_price.png")


def plot_log_returns(df, asset):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["timestamp"], df["log_return"], linewidth=0.7, color="tab:blue")
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.7)
    ax.set_title(f"{asset}: Log Returns Over Time")
    ax.set_xlabel("Time")
    ax.set_ylabel("Log return")
    save_figure(fig, FIGURE_DIR / f"{asset}_02_log_returns.png")


def plot_rolling_volatility_thresholds(df, asset, thresholds):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["timestamp"], df["rolling_realized_vol"], linewidth=1.0, color="tab:purple", label="Rolling realized volatility")
    ax.axhline(thresholds["vol_q33"], color="tab:green", linestyle="--", linewidth=1.2, label="q33 threshold")
    ax.axhline(thresholds["vol_q66"], color="tab:red", linestyle="--", linewidth=1.2, label="q66 threshold")
    ax.set_title(f"{asset}: Rolling Realized Volatility and Quantile Thresholds")
    ax.set_xlabel("Time")
    ax.set_ylabel("Rolling realized volatility")
    ax.legend()
    save_figure(fig, FIGURE_DIR / f"{asset}_03_rolling_volatility_quantile_thresholds.png")


def plot_price_colored_by_regime(df, asset):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["timestamp"], df["close"], color="lightgray", linewidth=0.8, label="Close price")
    for label in REGIME_ORDER:
        group = df[df["regime_quantile_label"] == label]
        ax.scatter(group["timestamp"], group["close"], s=7, color=REGIME_COLORS[label], label=label, alpha=0.75)
    ax.set_title(f"{asset}: Close Price Colored by Volatility-Quantile Regime")
    ax.set_xlabel("Time")
    ax.set_ylabel("Close price")
    ax.legend(markerscale=2)
    save_figure(fig, FIGURE_DIR / f"{asset}_04_price_colored_by_quantile_regime.png")


def plot_volatility_colored_by_regime(df, asset):
    fig, ax = plt.subplots(figsize=(12, 5))
    for label in REGIME_ORDER:
        group = df[df["regime_quantile_label"] == label]
        ax.scatter(group["timestamp"], group["rolling_realized_vol"], s=7, color=REGIME_COLORS[label], label=label, alpha=0.75)
    ax.set_title(f"{asset}: Rolling Volatility Colored by Volatility-Quantile Regime")
    ax.set_xlabel("Time")
    ax.set_ylabel("Rolling realized volatility")
    ax.legend(markerscale=2)
    save_figure(fig, FIGURE_DIR / f"{asset}_05_volatility_colored_by_quantile_regime.png")


def plot_regime_distribution(df, asset):
    counts = df[df["regime_quantile_label"].isin(REGIME_ORDER)]["regime_quantile_label"].value_counts().reindex(REGIME_ORDER, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(counts.index, counts.values, color=[REGIME_COLORS[label] for label in counts.index])
    ax.set_title(f"{asset}: Regime Distribution")
    ax.set_xlabel("Regime")
    ax.set_ylabel("Number of observations")
    for index, value in enumerate(counts.values):
        ax.text(index, value, f"{value:,}", ha="center", va="bottom", fontsize=9)
    save_figure(fig, FIGURE_DIR / f"{asset}_06_regime_distribution.png")


def plot_regime_timeline_strip(df, asset):
    valid = df[df["regime_quantile_id"].notna()].copy()
    if valid.empty:
        warnings.warn(f"{asset}: no valid regimes for timeline strip.")
        return

    x = valid["timestamp"]
    y = valid["regime_quantile_id"].astype(int).to_numpy().reshape(1, -1)
    fig, ax = plt.subplots(figsize=(12, 2.4))
    cmap = plt.matplotlib.colors.ListedColormap([REGIME_COLORS[label] for label in REGIME_ORDER])
    ax.imshow(y, aspect="auto", interpolation="nearest", cmap=cmap, vmin=0, vmax=2, extent=[0, len(valid), 0, 1])
    tick_positions = np.linspace(0, len(valid) - 1, min(8, len(valid))).astype(int)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([str(x.iloc[position].date()) for position in tick_positions], rotation=30, ha="right")
    ax.set_yticks([])
    ax.set_title(f"{asset}: Quantile Regime Timeline Strip")
    ax.set_xlabel("Time")
    handles = [plt.Line2D([0], [0], color=REGIME_COLORS[label], linewidth=6, label=label) for label in REGIME_ORDER]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3)
    save_figure(fig, FIGURE_DIR / f"{asset}_07_regime_timeline_strip.png")


def create_asset_figures(df, asset, thresholds):
    plot_close_price(df, asset)
    plot_log_returns(df, asset)
    plot_rolling_volatility_thresholds(df, asset, thresholds)
    plot_price_colored_by_regime(df, asset)
    plot_volatility_colored_by_regime(df, asset)
    plot_regime_distribution(df, asset)
    plot_regime_timeline_strip(df, asset)


for asset, df in regime_asset_data.items():
    create_asset_figures(df, asset, quantile_thresholds[asset])
    print(f"Saved figures for {asset} to {FIGURE_DIR}")

Saved figures for BTCUSDT to C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\figures


Saved figures for ETHUSDT to C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\figures


## 8. Interpretation Prompts and Auto-Generated Diagnostics

The diagnostics below summarize the empirical thresholds, regime proportions, separation in absolute returns and volatility, and whether regimes appear as persistent blocks or scattered labels.

In [8]:
def interpret_asset_regimes(asset, df, thresholds, summary_table, run_table):
    valid = df[df["regime_quantile_label"].isin(REGIME_ORDER)].copy()
    percentages = (
        valid["regime_quantile_label"]
        .value_counts(normalize=True)
        .reindex(REGIME_ORDER, fill_value=0)
        .mul(100)
    )

    mean_abs = summary_table.set_index("regime_quantile_label")["mean_abs_log_return"].to_dict()
    mean_vol = summary_table.set_index("regime_quantile_label")["mean_rolling_vol"].to_dict()
    high_separated = (
        mean_abs.get("high_vol", np.nan) > mean_abs.get("medium_vol", np.nan) > mean_abs.get("low_vol", np.nan)
        and mean_vol.get("high_vol", np.nan) > mean_vol.get("medium_vol", np.nan) > mean_vol.get("low_vol", np.nan)
    )

    average_run = run_table["mean_run_length"].mean() if not run_table.empty else np.nan
    persistence_text = "persistent blocks" if pd.notna(average_run) and average_run >= 3 else "frequent switching or scattered labels"

    print(f"\n{asset} interpretation")
    print("-" * (len(asset) + 15))
    print(f"Volatility q33: {thresholds['vol_q33']:.8f}")
    print(f"Volatility q66: {thresholds['vol_q66']:.8f}")
    for label in REGIME_ORDER:
        print(f"{label}: {percentages[label]:.2f}% of valid observations")
    print(f"High-volatility separation: {'yes' if high_separated else 'not strictly monotonic; inspect summary table'}")
    print(f"Regime persistence diagnostic: labels appear as {persistence_text} based on mean run length.")


for asset, df in regime_asset_data.items():
    asset_summary = regime_summary_by_asset[regime_summary_by_asset["asset"] == asset]
    asset_runs = regime_run_summary_by_asset[regime_run_summary_by_asset["asset"] == asset]
    interpret_asset_regimes(asset, df, quantile_thresholds[asset], asset_summary, asset_runs)


BTCUSDT interpretation
----------------------
Volatility q33: 0.00300329
Volatility q66: 0.00458023
low_vol: 33.00% of valid observations
medium_vol: 33.00% of valid observations
high_vol: 34.00% of valid observations
High-volatility separation: yes
Regime persistence diagnostic: labels appear as persistent blocks based on mean run length.

ETHUSDT interpretation
----------------------
Volatility q33: 0.00519710
Volatility q66: 0.00761209
low_vol: 33.00% of valid observations
medium_vol: 33.00% of valid observations
high_vol: 34.00% of valid observations
High-volatility separation: yes
Regime persistence diagnostic: labels appear as persistent blocks based on mean run length.


## How to Read These Plots

The rolling volatility plot shows the regime driver: recent realized variation in log returns. The q33 and q66 threshold lines define the low, medium, and high volatility states.

The price-colored plot shows where each regime occurs in market history. This helps connect volatility states to visible market episodes instead of treating labels as abstract categories.

The timeline strip shows persistence and switching. Long same-color blocks indicate stable volatility states, while rapid alternation suggests noisy switching.

The regime distribution chart checks whether segmentation is reasonably balanced. Since empirical quantiles are used, the valid observations should be roughly split into thirds, apart from ties and missing rolling-history rows.

The run-length table checks whether regimes form meaningful consecutive blocks instead of isolated point labels.

## 9. Save Regime Labels, Configuration, and Logs

The saved labels are intentionally clean and narrow so later motif notebooks can join them without inheriting unrelated feature columns.

In [9]:
label_output_paths = {}

for asset, df in regime_asset_data.items():
    label_columns = [
        "timestamp",
        "close",
        "log_return",
        "abs_log_return",
        "rolling_realized_vol",
        "regime_quantile_id",
        "regime_quantile_label",
    ]
    if "volume" in df.columns:
        label_columns.insert(2, "volume")

    labels = df[label_columns].copy()
    parquet_path = REGIME_LABEL_DIR / f"{asset}_quantile_regime_labels.parquet"
    csv_path = REGIME_LABEL_DIR / f"{asset}_quantile_regime_labels.csv"
    labels.to_parquet(parquet_path, index=False)
    labels.to_csv(csv_path, index=False)
    label_output_paths[asset] = {"parquet": str(parquet_path), "csv": str(csv_path)}
    print(f"Saved labels for {asset}: {parquet_path}")

config = {
    "target_symbols": TARGET_SYMBOLS,
    "assets_processed": list(regime_asset_data.keys()),
    "use_year_filter": USE_YEAR_FILTER,
    "year_analyzed": YEAR_TO_ANALYZE,
    "rolling_window_used_per_asset": rolling_windows_used,
    "lower_quantile": LOWER_QUANTILE,
    "upper_quantile": UPPER_QUANTILE,
    "input_files_used": input_files_used,
    "output_directory": str(RESULT_DIR),
    "created_timestamp": datetime.now(timezone.utc).isoformat(),
    "no_future_looking_features_used": True,
    "regime_driver": "rolling_realized_vol computed from current and historical log returns only",
}

config_path = CONFIG_DIR / "quantile_regime_config.json"
with open(config_path, "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)

discovery_log.to_csv(LOG_DIR / "data_discovery_log.csv", index=False)

print(f"Saved config: {config_path}")
print(f"Saved discovery log: {LOG_DIR / 'data_discovery_log.csv'}")

Saved labels for BTCUSDT: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\regime_labels\BTCUSDT_quantile_regime_labels.parquet


Saved labels for ETHUSDT: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\regime_labels\ETHUSDT_quantile_regime_labels.parquet
Saved config: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\config\quantile_regime_config.json
Saved discovery log: C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\reports\results\regime_studies\01_volatility_quantile_regimes\logs\data_discovery_log.csv


## 10. Thesis-Ready Dynamic Conclusion

The cell below generates a conclusion from the processed assets and saved regime summaries.

In [10]:
from IPython.display import Markdown, display

processed_assets = ", ".join(regime_asset_data.keys())
asset_lines = []
for asset in regime_asset_data:
    summary = regime_summary_by_asset[regime_summary_by_asset["asset"] == asset]
    high_row = summary[summary["regime_quantile_label"] == "high_vol"]
    low_row = summary[summary["regime_quantile_label"] == "low_vol"]
    if not high_row.empty and not low_row.empty:
        ratio = high_row["mean_abs_log_return"].iloc[0] / low_row["mean_abs_log_return"].iloc[0]
        asset_lines.append(f"- {asset}: high-volatility mean absolute log return was {ratio:.2f}x the low-volatility value.")

conclusion_md = f"""
### Conclusion

This notebook implemented volatility-quantile regime detection as the transparent statistical baseline for the thesis. Rolling realized volatility was used as the regime driver, and empirical 33rd and 66th percentile thresholds separated the market into low, medium, and high-volatility states. These labels will be used in later notebooks to compare regime-agnostic motif discovery against regime-conditioned motif discovery using Matrix Profile and LoCoMotif.

**Assets processed:** {processed_assets}

**What worked**

- The notebook discovered and loaded available target asset data from the recursive dataset directory.
- Rolling realized volatility produced deterministic low, medium, and high volatility labels.
- Summary tables, run-length diagnostics, figures, regime label files, config, and discovery logs were saved under the result directory.
{chr(10).join(asset_lines)}

**Limitations that remain**

- Quantile regimes are simple and interpretable, but they only use volatility.
- They do not learn transition probabilities.
- They do not model hidden persistence like an HMM.
- Thresholds are empirical and sample-dependent, so regime boundaries can shift when the analysis window changes.

**Connection to the next notebook**

The next regime notebook should add the HMM-based method and compare its learned state persistence with this transparent quantile baseline. Motif discovery should remain separate and use the saved regime labels from this notebook as inputs.
"""

display(Markdown(conclusion_md))


### Conclusion

This notebook implemented volatility-quantile regime detection as the transparent statistical baseline for the thesis. Rolling realized volatility was used as the regime driver, and empirical 33rd and 66th percentile thresholds separated the market into low, medium, and high-volatility states. These labels will be used in later notebooks to compare regime-agnostic motif discovery against regime-conditioned motif discovery using Matrix Profile and LoCoMotif.

**Assets processed:** BTCUSDT, ETHUSDT

**What worked**

- The notebook discovered and loaded available target asset data from the recursive dataset directory.
- Rolling realized volatility produced deterministic low, medium, and high volatility labels.
- Summary tables, run-length diagnostics, figures, regime label files, config, and discovery logs were saved under the result directory.
- BTCUSDT: high-volatility mean absolute log return was 2.55x the low-volatility value.
- ETHUSDT: high-volatility mean absolute log return was 2.33x the low-volatility value.

**Limitations that remain**

- Quantile regimes are simple and interpretable, but they only use volatility.
- They do not learn transition probabilities.
- They do not model hidden persistence like an HMM.
- Thresholds are empirical and sample-dependent, so regime boundaries can shift when the analysis window changes.

**Connection to the next notebook**

The next regime notebook should add the HMM-based method and compare its learned state persistence with this transparent quantile baseline. Motif discovery should remain separate and use the saved regime labels from this notebook as inputs.


## 11. Final Validation

This validation cell verifies that the notebook processed at least one asset, saved the expected regime labels, created summary tables, produced figures, avoided future-looking features, and assigned nonzero observations to low, medium, and high regimes where possible.

In [11]:
def validate_quantile_regime_notebook():
    checks = []

    checks.append((len(regime_asset_data) >= 1, "At least one asset processed"))

    for asset in regime_asset_data:
        parquet_path = REGIME_LABEL_DIR / f"{asset}_quantile_regime_labels.parquet"
        csv_path = REGIME_LABEL_DIR / f"{asset}_quantile_regime_labels.csv"
        checks.append((parquet_path.exists(), f"Parquet labels exist for {asset}"))
        checks.append((csv_path.exists(), f"CSV labels exist for {asset}"))

        figure_count = len(list(FIGURE_DIR.glob(f"{asset}_*.png")))
        checks.append((figure_count >= 5, f"At least 5 figures exist for {asset}"))

        valid_counts = regime_asset_data[asset]["regime_quantile_label"].value_counts()
        for label in REGIME_ORDER:
            checks.append((valid_counts.get(label, 0) > 0, f"{asset} has nonzero {label} observations"))

    checks.append(((TABLE_DIR / "regime_summary_by_asset.csv").exists(), "Regime summary table exists"))
    checks.append(((TABLE_DIR / "regime_run_summary_by_asset.csv").exists(), "Regime run summary table exists"))
    checks.append(((LOG_DIR / "data_discovery_log.csv").exists(), "Data discovery log exists"))
    checks.append(((CONFIG_DIR / "quantile_regime_config.json").exists(), "Config JSON exists"))
    checks.append((config.get("no_future_looking_features_used") is True, "No future-looking features were used"))

    validation_df = pd.DataFrame(checks, columns=["passed", "check"])
    display(validation_df)

    failed = validation_df[~validation_df["passed"]]
    if not failed.empty:
        raise AssertionError("Validation failed:\n" + failed.to_string(index=False))

    print("QUANTILE REGIME NOTEBOOK COMPLETED SUCCESSFULLY")


validate_quantile_regime_notebook()

,passed,check
0,True,At least one asset processed
1,True,Parquet labels exist for BTCUSDT
2,True,CSV labels exist for BTCUSDT
3,True,At least 5 figures exist for BTCUSDT
4,True,BTCUSDT has nonzero low_vol observations
5,True,BTCUSDT has nonzero medium_vol observations
6,True,BTCUSDT has nonzero high_vol observations
7,True,Parquet labels exist for ETHUSDT
8,True,CSV labels exist for ETHUSDT
9,True,At least 5 figures exist for ETHUSDT


QUANTILE REGIME NOTEBOOK COMPLETED SUCCESSFULLY
